# Functions and Loops: One Analysis, Many Inputs

**Series:** Automating Repeated Steps in Python (notebook 1 of 2)
**Time:** about 30 minutes

**In this notebook you will:**
- Turn a sequence of geoprocessing steps into a function with parameters
- Check that the function reproduces a result you already know
- Loop over distances and over facility layers, and collect the results in a table
- Guard a function against a common mistake (a CRS in degrees)

**You should already know:** functions (Foundations notebook <a href="https://colab.research.google.com/github/SpatialTurn/Foundations/blob/main/notebooks/02f_Functions.ipynb">2f</a>) and the Geoprocessing in Python episode (notebooks 02a to 02c).

> **Before you start:** click **File > Save a copy in Drive** so your changes are kept.
> Then run each cell in order by clicking it and pressing **Shift + Enter**.

## Why automate?

In the Geoprocessing episode you estimated that about **77,500** of the **114,000** residents of Lafayette and
West Lafayette live within 1 mile of a supermarket. Then the questions start: what about half a mile? Two
miles? Any food store rather than supermarkets only? Parks instead of stores? Each is the same seven steps with
one input changed. Copying the cell seven times works once, but every copy is a chance to change one line and
forget another.

A **function** writes the steps once, with the parts that change as **parameters**. A **loop** then runs it for
every input you care about.

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

stores = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/CaseStudies/main/data/grocery_stores.geojson").to_crs(epsg=26916)
city = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/CaseStudies/main/data/study_area.geojson").to_crs(epsg=26916)
tracts = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_tracts_acs.geojson").to_crs(epsg=26916)
parks = gpd.read_file("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_parks.geojson").to_crs(epsg=26916)
supermarkets = stores[stores["store_type"] == "Supermarket"]
print(len(supermarkets), "supermarkets,", len(stores), "food stores,", len(parks), "parks,", len(tracts), "tracts")

## Step 1: Write the function

These are the steps of notebook 02b, unchanged, inside a `def`. The **docstring** (the text in triple quotes)
says what the function does, what it needs and what it returns. Read it back with `help(pop_within)`.

In [ ]:
def pop_within(facilities, distance, tracts=tracts, boundary=city, column="total_pop"):
    """Estimate how many people inside `boundary` live within `distance` meters of `facilities`.

    Uses areal weighting: each tract's `column` is split in proportion to area.
    All layers must be in a projected CRS measured in meters.
    Returns (people within the distance, people in the boundary, share in percent).
    """
    t = tracts.copy()
    t["tract_m2"] = t.area                                        # original tract area
    in_boundary = gpd.clip(t, boundary)
    zone = gpd.GeoDataFrame(geometry=[facilities.buffer(distance).union_all()], crs=t.crs)
    near = gpd.overlay(in_boundary, zone, how="intersection")
    people_near = (near.area / near["tract_m2"] * near[column]).sum()
    people_all = (in_boundary.area / in_boundary["tract_m2"] * in_boundary[column]).sum()
    return round(people_near), round(people_all), round(people_near / people_all * 100, 1)

`tracts=tracts`, `boundary=city` and `column="total_pop"` are **default values**: you can leave them out, or
pass something else. `facilities` and `distance` have no default, so every call must give them.

## Step 2: Check it against a known answer

Before trusting a function, run it on a case where you already know the answer.

In [ ]:
result = pop_within(supermarkets, 1609)
print(result)
assert result[0] == 77473, "does not match the Geoprocessing episode"

77,473 people, 67.8%: exactly the Geoprocessing episode's result. `assert` stops the notebook with a message if
the condition is false. It costs one line and catches the day a change breaks the function.

## Step 3: Loop over distances

In [ ]:
rows = []
for d in [400, 800, 1609, 3218]:
    near, total, share = pop_within(supermarkets, d)
    rows.append({"distance_m": d, "people_within": near, "share_pct": share})
by_distance = pd.DataFrame(rows)
by_distance

The pattern is the heart of most automation: **an empty list, a loop that appends one result per input, and a
table at the end.** A half-mile walk (800 m) reaches only about a quarter of residents; two miles reach almost
everyone.

In [ ]:
ax = by_distance.plot(x="distance_m", y="share_pct", marker="o", legend=False, figsize=(6, 4))
ax.set_xlabel("Distance to the nearest supermarket (m, straight line)")
ax.set_ylabel("Residents within the distance (%)")
ax.set_ylim(0, 100)
plt.show()

## Step 4: Loop over facility layers

The same function works for any layer of points or polygons. A dictionary keeps a name with each layer.

In [ ]:
layers = {"Supermarkets": supermarkets, "All food stores": stores, "Parks": parks}
rows = []
for name, layer in layers.items():
    for d in [800, 1609]:
        near, total, share = pop_within(layer, d)
        rows.append({"facilities": name, "distance_m": d, "share_pct": share})
pd.DataFrame(rows).pivot(index="facilities", columns="distance_m", values="share_pct")

Parks are buffered from their edges (a polygon buffer grows outward from the boundary), so 71.5% of residents
live within 800 m of one of the 68 OpenStreetMap parks; the Park Access case study refines this with public parks
only. Two nested loops ran six analyses in a few seconds.

## Step 5: Guard against a common mistake

What happens if someone passes a layer still in latitude and longitude? A buffer of 1,609 *degrees* covers the
whole planet, and the function would happily report that everyone lives near a store. Add a check that stops
with a clear message instead:

In [ ]:
def check_meters(*layers):
    """Raise an error unless every layer is in a projected CRS measured in meters."""
    for layer in layers:
        if layer.crs is None or not layer.crs.is_projected:
            raise ValueError(f"CRS {layer.crs} is not projected: reproject with .to_crs(epsg=26916) first")

check_meters(supermarkets, tracts, city)       # fine: no output
try:
    check_meters(supermarkets.to_crs(epsg=4326))
except ValueError as e:
    print("Stopped:", e)

In your own function, call `check_meters(facilities, tracts, boundary)` on its first line. `try` and `except`
catch the error here so the notebook can continue; without them the error stops the notebook, which is what you
want when a real mistake happens.

## Try it yourself

The function has a `column` parameter. Use it to estimate the share of **children** (`pop_under18`) within
1 mile of a supermarket, and compare it with the share of all residents (67.8%).

*Hint:* Pass `column="pop_under18"` to `pop_within`.

<details>
<summary><b>Show a solution</b></summary>

```python
near, total, share = pop_within(supermarkets, 1609, column="pop_under18")
print(f"{near:,} of {total:,} children ({share}%)")
```

</details>

In [ ]:
# Write your answer here

## Recap

- A function names a sequence of steps; parameters are the parts that change, with defaults for the usual case
- Test a function on a known answer, and keep the test with `assert`
- Loop pattern: empty list, append one result per input, build a table at the end
- Check inputs at the start of a function and fail with a clear message

**Next:** [04b Many Counties: Automating a Workflow](https://colab.research.google.com/github/SpatialTurn/DataProcessing/blob/main/notebooks/04b_Many_Counties.ipynb)